# 04 - Fall detection: model, then calibration

| Setting | Value |
| --- | --- |
| Internet | **Off** |
| Accelerator | **GPU** |

The deployed fall path is **rule-first**: `wellbeing.activity.fall.FallDetector` fires on
geometry (rapid centroid drop, sustained horizontal torso, post-event stillness) and this
learned model only *confirms* it. That ordering is deliberate. A rule that fires on geometry is
explainable to a caregiver and to an incident review; a model that fires on an embedding is not.

So this notebook does two things, and the second matters more:

1. Train a confirmation classifier on rule-triggered candidate windows.
2. **Choose the threshold against a false-alarm budget of one alarm per resident-week.**

Reporting ROC-AUC here would be close to meaningless: at a 1-in-100,000-windows event rate,
AUC 0.99 still yields dozens of nightly false alarms, and a fall detector that gets muted has
an effective recall of zero.


In [ ]:
import json, socket, sys
from pathlib import Path

import numpy as np
import torch

assert torch.cuda.is_available(), 'Enable the GPU accelerator in the notebook settings.'
print('gpu:', torch.cuda.get_device_name(0))

def has_internet(host='raw.githubusercontent.com', port=443, timeout=3):
    try:
        socket.create_connection((host, port), timeout=timeout)
        return True
    except OSError:
        return False

print('internet:', 'ENABLED - turn it off' if has_internet() else 'disabled (correct)')

DATASET_DIR = Path('/kaggle/input/fall-lowres-v1')
RUN_DIR = Path('/kaggle/working/runs/fall-001')
assert DATASET_DIR.exists(), f'attach the prepared fall dataset at {DATASET_DIR}'

sys.path.insert(0, str(DATASET_DIR / 'code' / 'src'))
sys.path.insert(0, str(DATASET_DIR / 'code' / 'train'))

manifest = json.loads((DATASET_DIR / 'manifest.json').read_text())
records = manifest['records']
print(f"dataset: {manifest['name']} v{manifest['version']}, {manifest['n_records']} clips")


In [ ]:
from _offline_tracker import OfflineTracker
from train_fall import (
    MAX_FALSE_ALARMS_PER_RESIDENT_WEEK, TARGET_RECALL, seed_everything, threshold_for_budget,
)

SEED = 42
EPOCHS = 25
BATCH_SIZE = 32
LR = 5e-4
WINDOW_FRAMES = 48  # ~2s at 24fps: covers the drop plus the stillness that follows it

seed_everything(SEED)
print(f'target recall {TARGET_RECALL}, budget {MAX_FALSE_ALARMS_PER_RESIDENT_WEEK} false alarms/resident-week')

# How many hours of genuine negative footage the test split represents. This drives the whole
# calibration and is the single easiest number to get wrong: it must be wall-clock hours of
# ordinary living, not the duration of the curated negative clips.
NEGATIVE_HOURS = 24 * 7 * 8
print(f'negative footage: {NEGATIVE_HOURS}h = {NEGATIVE_HOURS / (24 * 7):.1f} resident-weeks')


## Features: pose geometry, not raw pixels

The confirmation model consumes the same geometric time series the rules do - centroid height,
body height, torso angle, speed - rather than video frames. Three reasons, in order of
importance:

1. **It stays explainable.** Feature attributions map onto phrases a caregiver understands.
2. **It transfers across cameras.** Pixel models overfit one room's furniture and lighting.
3. **It runs on CPU at the edge**, so the safety path survives GPU loss (see the degradation
   ladder in `configs/default.yaml`).


In [ ]:
FEATURES = ['centroid_y_norm', 'body_height_norm', 'torso_angle_deg', 'speed_px_s', 'aspect_ratio']

def window_from_record(record):
    """Return a (WINDOW_FRAMES, len(FEATURES)) array for one clip.

    Replace the synthetic branch with your prepared pose track. Keeping the shape contract here
    means the training code does not change when real tracks arrive.
    """
    track_path = DATASET_DIR / record['path']
    if track_path.suffix == '.npy' and track_path.exists():
        series = np.load(track_path).astype(np.float32)
    else:
        rng = np.random.default_rng(abs(hash(record['path'])) % (2 ** 32))
        series = np.zeros((WINDOW_FRAMES, len(FEATURES)), dtype=np.float32)
        is_fall = record['label'] == 'fall'
        impact = rng.integers(12, 24) if is_fall else WINDOW_FRAMES + 1
        for t in range(WINDOW_FRAMES):
            after = t >= impact
            series[t, 0] = (0.85 if after else 0.35) + rng.normal(0, 0.02)  # centroid drops
            series[t, 1] = (0.25 if after else 0.60) + rng.normal(0, 0.02)  # body foreshortens
            series[t, 2] = (78.0 if after else 12.0) + rng.normal(0, 4.0)   # torso goes horizontal
            series[t, 3] = (2.0 if after else 30.0) + rng.normal(0, 3.0)    # then stillness
            series[t, 4] = (2.4 if after else 0.45) + rng.normal(0, 0.05)
        if not is_fall and record['label'] in ('negative_sit', 'negative_lie'):
            # Hard negatives: sitting and lying down look like the end state of a fall. The only
            # separating signal is how fast the transition happened.
            start = rng.integers(8, 30)
            for t in range(start, WINDOW_FRAMES):
                progress = min(1.0, (t - start) / 20.0)  # gradual, not sudden
                series[t, 0] = 0.35 + 0.45 * progress
                series[t, 2] = 12.0 + 62.0 * progress
                series[t, 3] = max(1.5, 30.0 - 25.0 * progress)
    if len(series) >= WINDOW_FRAMES:
        return series[:WINDOW_FRAMES]
    pad = np.repeat(series[-1:], WINDOW_FRAMES - len(series), axis=0)
    return np.concatenate([series, pad])

def build_split(split):
    rows = [r for r in records if r.get('split') == split]
    if not rows:
        return None, None, []
    x = np.stack([window_from_record(r) for r in rows])
    y = np.array([1 if r['label'] == 'fall' else 0 for r in rows], dtype=np.float32)
    return x, y, rows

x_train, y_train, train_rows = build_split('train')
x_val, y_val, _ = build_split('val')
x_test, y_test, test_rows = build_split('test')
assert x_train is not None, 'no train split in the manifest'

# Normalise using TRAIN statistics only. Fitting the scaler on all data leaks test information
# and inflates every number below.
mean = x_train.reshape(-1, len(FEATURES)).mean(0)
std = x_train.reshape(-1, len(FEATURES)).std(0) + 1e-6
normalise = lambda a: None if a is None else (a - mean) / std
x_train, x_val, x_test = normalise(x_train), normalise(x_val), normalise(x_test)

print(f'train {x_train.shape}, positives {int(y_train.sum())}/{len(y_train)}')
if x_test is not None:
    print(f'test  {x_test.shape}, positives {int(y_test.sum())}/{len(y_test)}')


In [ ]:
# Small temporal CNN. Kept deliberately small: the dataset is a few thousand windows, and a
# larger model would memorise the handful of fall subjects rather than learn the dynamics.
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

class FallScorer(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(n_features, 64, 5, padding=2), nn.BatchNorm1d(64), nn.ReLU(),
            nn.Conv1d(64, 64, 5, padding=2, dilation=1), nn.BatchNorm1d(64), nn.ReLU(),
            nn.Conv1d(64, 64, 3, padding=2, dilation=2), nn.BatchNorm1d(64), nn.ReLU(),
            nn.AdaptiveMaxPool1d(1), nn.Flatten(), nn.Dropout(0.3), nn.Linear(64, 1),
        )

    def forward(self, x):
        return self.net(x.transpose(1, 2)).squeeze(1)

def as_loader(x, y, shuffle):
    dataset = TensorDataset(torch.from_numpy(x).float(), torch.from_numpy(y).float())
    return DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=shuffle)

model = FallScorer(len(FEATURES)).cuda()
train_loader = as_loader(x_train, y_train, True)
val_loader = as_loader(x_val, y_val, False) if x_val is not None else None

# Positive weighting matters: falls are ~1 in 5 here but ~1 in 100k in deployment. Weight for
# recall now and control precision with the threshold later, where it is auditable.
pos_weight = torch.tensor([(len(y_train) - y_train.sum()) / max(1.0, y_train.sum())]).cuda()
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
print('pos_weight:', float(pos_weight))


In [ ]:
tracker = OfflineTracker(RUN_DIR, {
    'task': 'fall_confirmation',
    'dataset': manifest['name'],
    'dataset_version': manifest['version'],
    'features': FEATURES,
    'window_frames': WINDOW_FRAMES,
    'negative_hours': NEGATIVE_HOURS,
    'hyperparameters': {'seed': SEED, 'epochs': EPOCHS, 'batch_size': BATCH_SIZE, 'lr': LR},
})

@torch.no_grad()
def score(loader):
    model.eval()
    scores, truth = [], []
    for windows, targets in loader:
        scores.append(torch.sigmoid(model(windows.cuda())).cpu().numpy())
        truth.append(targets.numpy())
    return np.concatenate(scores), np.concatenate(truth)

# Model selection uses recall at a fixed low false-positive rate, not accuracy or AUC.
best_metric, best_state = -1.0, None
for epoch in range(EPOCHS):
    model.train()
    running = 0.0
    for windows, targets in train_loader:
        optimizer.zero_grad(set_to_none=True)
        loss = criterion(model(windows.cuda()), targets.cuda())
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        running += float(loss)

    metric = float('nan')
    if val_loader is not None:
        scores, truth = score(val_loader)
        negatives = scores[truth == 0]
        if len(negatives) > 0 and truth.sum() > 0:
            # Recall at the 99th percentile of negative scores: a proxy for the deployed operating point.
            cut = float(np.quantile(negatives, 0.99))
            metric = float((scores[truth == 1] >= cut).mean())
        tracker.log(epoch, train_loss=running / max(1, len(train_loader)), recall_at_1pct_fpr=metric)
        if metric > best_metric:
            best_metric = metric
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    if epoch % 5 == 0 or epoch == EPOCHS - 1:
        print(f'epoch {epoch:02d}  loss {running / max(1, len(train_loader)):.4f}  recall@1%FPR {metric:.4f}')

if best_state is not None:
    model.load_state_dict(best_state)
print('best recall@1%FPR:', best_metric)


## Calibration: the cell that decides whether this ships

`threshold_for_budget` walks candidate thresholds and returns the lowest one whose projected
false-alarm rate still fits inside the budget. Latency is checked alongside it, because a
correct alert that arrives twenty minutes late has not helped anyone off the floor.


In [ ]:
assert x_test is not None, 'no test split; calibrating on val would leak and overstate precision'
test_scores, test_truth = score(as_loader(x_test, y_test, False))
paired = list(zip(test_scores.tolist(), [bool(v) for v in test_truth.tolist()]))

threshold, recall, false_per_week = threshold_for_budget(paired, NEGATIVE_HOURS)
weeks = NEGATIVE_HOURS / (24 * 7)

print(f'threshold             : {threshold:.4f}')
print(f'recall                : {recall:.4f}   (target >= {TARGET_RECALL})')
print(f'false alarms/res-week : {false_per_week:.3f}   (budget <= {MAX_FALSE_ALARMS_PER_RESIDENT_WEEK})')
print()
print(f"{'threshold':>10}{'recall':>10}{'FA/week':>10}{'precision':>11}")
for candidate in [0.5, 0.6, 0.7, 0.8, 0.9, 0.95, threshold]:
    fired = test_scores >= candidate
    true_positives = int((fired & (test_truth == 1)).sum())
    false_positives = int((fired & (test_truth == 0)).sum())
    candidate_recall = true_positives / max(1, int(test_truth.sum()))
    precision = true_positives / max(1, true_positives + false_positives)
    marker = '  <- chosen' if abs(candidate - threshold) < 1e-9 else ''
    print(f'{candidate:>10.3f}{candidate_recall:>10.3f}{false_positives / weeks:>10.2f}{precision:>11.3f}{marker}')


In [ ]:
# Where the residual errors are. Misses on unwitnessed night falls are far more serious than
# misses on a slow slide out of a chair in a monitored room, and an aggregate recall number
# hides that difference entirely.
fired = test_scores >= threshold
misses = [test_rows[i] for i in range(len(test_rows)) if test_truth[i] == 1 and not fired[i]]
false_alarms = [test_rows[i] for i in range(len(test_rows)) if test_truth[i] == 0 and fired[i]]

print(f'{len(misses)} missed falls:')
for row in misses[:10]:
    print(f"  {row['subject_id']:<16} {row.get('source', '?'):<12} {row['path']}")

print()
from collections import Counter
print(f'{len(false_alarms)} false alarms by true label:')
for label, count in Counter(r['label'] for r in false_alarms).most_common():
    print(f'  {label:<24} {count}')

met_recall = recall >= TARGET_RECALL
met_budget = false_per_week <= MAX_FALSE_ALARMS_PER_RESIDENT_WEEK
tracker.summarise(
    threshold=threshold,
    recall=recall,
    false_alarms_per_resident_week=false_per_week,
    negative_hours=NEGATIVE_HOURS,
    n_missed=len(misses),
    false_alarm_labels=dict(Counter(r['label'] for r in false_alarms)),
    feature_normalisation={'mean': mean.tolist(), 'std': std.tolist()},
    exit_criteria={'recall': TARGET_RECALL, 'fa_per_resident_week': MAX_FALSE_ALARMS_PER_RESIDENT_WEEK,
                   'met': bool(met_recall and met_budget)},
)
torch.save(
    {'model': model.state_dict(), 'threshold': threshold, 'features': FEATURES,
     'mean': mean.tolist(), 'std': std.tolist(), 'window_frames': WINDOW_FRAMES},
    RUN_DIR / 'fall_scorer.pt',
)
print()
print('recall gate:', 'MET' if met_recall else 'NOT MET')
print('false-alarm budget:', 'MET' if met_budget else 'NOT MET')
print('saved', RUN_DIR / 'fall_scorer.pt')


## Wiring the result into the running system

Set `activity.fall.model_confidence_floor` in `configs/default.yaml` to the threshold above.
The model confirms candidates that the geometric rules already raised; it never raises a
candidate on its own, so a model regression degrades confidence rather than blinding the system.

**If the budget cannot be met**, in order of preference:

1. **Mine hard negatives.** Sitting down heavily, lying on a sofa, bending to pick something up,
   and a pet crossing the frame produce most real-world false alarms. Twenty hard negatives beat
   two thousand easy ones.
2. **Require a longer stillness hold.** Raising `post_event_stillness` from 10s costs latency but
   removes recovered stumbles, which are not the events worth waking a caregiver for.
3. **Route medium-confidence detections to a check-in prompt instead of a critical alarm.** The
   contracts already support this: `Severity.WARNING` does not page anyone, and
   `Alert.suppressed_reason` records why the escalation was withheld.

What not to do: ship at three false alarms per week and assume caregivers will tolerate it.
They will mute the alerts within a fortnight, and then the detector's real recall is zero while
the dashboard still claims coverage.
